# Workshop: Gemma from Scratch
## Notebook 3: Grouped Query Attention (GQA)

**Estimated Time: 15 minutes**

Standard Multi-Head Attention (MHA) gives every Query head its own Key and Value head. This is memory-intensive for large models. **Grouped Query Attention (GQA)** optimizes this by sharing one K and V head among multiple Query heads.

### Learning Objectives:
1. Contrast MHA, MQA (Multi-Query), and GQA.
2. Understand the efficiency gains of GQA.
3. Implement the tensor repetition logic used in GQA.

In [ ]:
import torch
import torch.nn as nn
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

batch_size = 1
seq_len = 8
num_heads = 8
num_kv_groups = 2
head_dim = 16

group_size = num_heads // num_kv_groups
print(f"Each KV head will be shared by {group_size} Query heads.")

### 1. Generating Q, K, V with Groups

Notice the shape of K and V: they have fewer heads than Q.

In [ ]:
Q = torch.randn(batch_size, num_heads, seq_len, head_dim)
K = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)
V = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)

print(f"Q heads: {Q.shape[1]}")
print(f"K heads: {K.shape[1]}")

### 2. The Repetition Trick

To compute attention, we need a K and V for every Q. We use `repeat_interleave` to "expand" our groups to match the number of Query heads.

In [ ]:
# Expand K and V
K_expanded = K.repeat_interleave(group_size, dim=1)
V_expanded = V.repeat_interleave(group_size, dim=1)

print(f"Expanded K heads: {K_expanded.shape[1]}")
assert K_expanded.shape[1] == Q.shape[1]

### 3. Why GQA?

In inference, we store K and V in a **KV Cache**. By using fewer KV heads, we drastically reduce the memory footprint of the cache, allowing for larger batch sizes and longer contexts.

### Exercise:
Complete the `SimpleGQA` class. It should project an input $x$ to $Q, K, V$, expand $K$ and $V$, and compute the attention output.

**Hints:**
1. In `__init__`, define `self.W_q`, `self.W_k`, and `self.W_v` as `nn.Linear` layers.
2. In `forward`, after projecting, reshape the tensors to `(B, num_heads, T, head_dim)`.
3. Use `repeat_interleave` on $K$ and $V$ to match $Q$'s number of heads.
4. Reuse your `scaled_dot_product_attention` logic.

In [ ]:
class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups
        
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)
        
    def forward(self, x):
        B, T, C = x.shape
        
        # 1. Project and Reshape
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        
        # 2. Expand K, V groups to match Q heads
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        
        # 3. Scaled Dot-Product Attention
        # Your code here:
        # scores = ...
        # weights = ...
        # out = ...
        
        # Placeholder logic (return zeros for now)
        out = torch.zeros_like(q)
        
        # 4. Reshape and Project Out
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)

# Test your implementation
d_in, n_h, n_kv, h_d = 32, 8, 2, 16
model = SimpleGQA(d_in, n_h, n_kv, h_d)
x = torch.randn(1, 5, d_in)
try:
    output = model(x)
    print(f"✅ Success! Output shape: {output.shape}")
except Exception as e:
    print(f"❌ Error: {e}")

<details>
<summary><b>Click to see solution</b></summary>

```python
    def forward(self, x):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = torch.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)
        
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)
```
</details>